# NLLB-200: Kiswahili, Somali, Dholuo - combined vs. independent, one notebook

**One notebook, both training designs, in order: combined first, then independent** -
trains a single multi-task checkpoint on all three target languages at once (Section 7),
then three separate per-language checkpoints (Section 8, one cell each). Both get
evaluated together in one pass (Section 9), compared to pick an overall winner
(Section 10), and **both sets of checkpoints are saved regardless of which wins**
(Section 12) - nothing gets discarded. Domain ablation (Section 11) then runs only on
whichever approach actually won, since that's the one going forward.

**Completely independent of `02_nllb_training.ipynb`/`BEST_NLLB_CHECKPOINT`** - both
training designs initialize from the raw `facebook/nllb-200-distilled-600M` checkpoint.

**Data**: `05_other_languages_preprocess.ipynb`'s three outputs - run that notebook
first, this one only reads them.

**How the winner is decided**: n-weighted chrF2++, aggregated across all three
languages' overall (non-domain-split) scores - same "don't average domain-means
unweighted" principle already established for the main NLLB ablation, applied here to
"combined" vs. "independent" as the two things being compared instead of two curricula.

## 1. Setup

In [1]:
!pip install -q torch transformers accelerate sentencepiece evaluate sacrebleu mlflow peft matplotlib pandas datasets


### GPU / CUDA check — run this before anything else

In [2]:
import torch
print(f"torch version       : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU visible to torch - fix this before continuing.")


torch version       : 2.11.0+cu128
CUDA available      : True
GPU                 : NVIDIA H100 PCIe


In [3]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")
if "final_training" not in str(PROJECT_ROOT).lower().replace(" ", "_"):
    print("\n  WARNING: 'Final_Training' doesn't appear in PROJECT_ROOT above - move "
          "this notebook under Final_Training/Notebooks/ before continuing.")

from arch_config import CHECKPOINTS_DIR, LOGS_DIR, DEPLOYMENT_DIR, DATA_DIR
import data_io, metrics, inference
from train import train_stage
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from datasets import Dataset
import pandas as pd

SEED = 42
set_seed(SEED)
MAX_LEN = 128
CHECKPOINT = "facebook/nllb-200-distilled-600M"   # raw base - genuinely independent
LORA_KWARGS = {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05}
OTHER_LANGS_DATA_DIR = Path(DATA_DIR) / "other_langs"

LANGUAGE_CODES = {"kiswahili": "swh_Latn", "somali": "som_Latn", "dholuo": "luo_Latn"}
DOMAINS = ["Agriculture", "Health", "Education", "Security", "Governance"]

OTHER_LANG_CFG = {
    "lora_target_modules": ["q_proj", "v_proj"],
    "freeze_layers": 6,
    "unfreeze_embeddings": False,
    "tgt_code": None,
    "related_code": None,
    "learning_rate": 5e-5,
    "batch_size": 32,
}
print("configured language codes (verified against the real tokenizer next):", LANGUAGE_CODES)


project root: /home/jovyan/PSA/Final_Training
configured language codes (verified against the real tokenizer next): {'kiswahili': 'swh_Latn', 'somali': 'som_Latn', 'dholuo': 'luo_Latn'}


## 2. Verify the language codes against the real NLLB tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
print("Checking each target language code against the real tokenizer:\n")
vocab = tokenizer.get_vocab()
verified = {}
for lang_name, code_guess in LANGUAGE_CODES.items():
    is_present = code_guess in vocab
    status = "CONFIRMED" if is_present else "NOT FOUND - do not trust this code"
    print(f"  {lang_name:12s} {code_guess:12s} {status}")
    verified[lang_name] = is_present


Checking each target language code against the real tokenizer:

  kiswahili    swh_Latn     CONFIRMED
  somali       som_Latn     CONFIRMED
  dholuo       luo_Latn     CONFIRMED


## 3. Load `05`'s three outputs

In [5]:
train_path = OTHER_LANGS_DATA_DIR / "other_langs_train.csv"
test_path = OTHER_LANGS_DATA_DIR / "other_langs_test.csv"
val_path = OTHER_LANGS_DATA_DIR / "other_langs_validation.csv"
for p in [train_path, test_path, val_path]:
    if not p.exists():
        raise FileNotFoundError(f"{p} not found - run 05_other_languages_preprocess.ipynb first.")

train_pool = pd.read_csv(train_path)
shared_test = pd.read_csv(test_path)
shared_val = pd.read_csv(val_path)
print(f"train pool: {len(train_pool):,} rows")
print(f"shared test set: {len(shared_test):,} rows")
print(f"shared validation set: {len(shared_val):,} rows")


train pool: 19,338 rows
shared test set: 985 rows
shared validation set: 1,589 rows


## 4. Build training records - per language, plus a combined concatenation

`records[lang]` (train/val/test) is used both by the independent training cells directly
and as the source material concatenated into `combined_train`/`combined_val` for the
combined training cell - built once, used both ways, not duplicated logic.

In [6]:
def build_records(df, src_col, tgt_col, tgt_code, corpus_label, domain_col=None):
    out = []
    for r in df.itertuples(index=False):
        src, tgt = str(getattr(r, src_col)).strip(), str(getattr(r, tgt_col)).strip()
        if src and src.lower() != "nan" and tgt and tgt.lower() != "nan":
            rec = {"src_lang": "eng_Latn", "tgt_lang": tgt_code, "src": src, "tgt": tgt,
                  "corpus": corpus_label}
            if domain_col is not None:
                rec["domain"] = getattr(r, domain_col)
            out.append(rec)
    return out

records = {}
for lang in ["kiswahili", "somali", "dholuo"]:
    if not verified.get(lang, False):
        print(f"skipping {lang} - code not verified (Section 2)")
        continue
    target_col = lang.capitalize()
    train_rows = build_records(train_pool, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target")
    val_rows = build_records(shared_val, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target")
    test_rows = build_records(shared_test, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target",
                              domain_col="psa_domain")
    records[lang] = (train_rows, val_rows, test_rows)
    print(f"{lang}_target: train={len(train_rows):,} | val={len(val_rows):,} | test={len(test_rows):,}")

combined_train = [r for lang in records for r in records[lang][0]]
combined_val = [r for lang in records for r in records[lang][1]]
print(f"\ncombined train (all languages): {len(combined_train):,} rows")
print(f"combined validation (all languages): {len(combined_val):,} rows")


kiswahili_target: train=19,338 | val=1,589 | test=985
somali_target: train=19,338 | val=1,589 | test=985
dholuo_target: train=19,338 | val=1,589 | test=985

combined train (all languages): 58,014 rows
combined validation (all languages): 4,767 rows


## 5. Tokenization - both the combined dataset and each independent one

The combined tokenizer reads `tgt_lang` per row (the dataset mixes all three target
languages); each independent tokenizer takes a single fixed `tgt_lang` per call, since
each only ever handles one language.

In [7]:
def nllb_preprocess_combined(batch):
    all_ids = []
    for sl, tl, src, tgt in zip(batch["src_lang"], batch["tgt_lang"], batch["src"], batch["tgt"]):
        tokenizer.src_lang = sl
        tokenizer.tgt_lang = tl
        enc = tokenizer(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
    return {"input_ids": [e["input_ids"] for e in all_ids],
            "attention_mask": [e["attention_mask"] for e in all_ids],
            "labels": [e["labels"] for e in all_ids]}

def tokenize_combined(rows):
    ds = Dataset.from_list(rows)
    return ds.map(nllb_preprocess_combined, batched=True, batch_size=16)

def nllb_preprocess_single(batch, tgt_lang):
    all_ids = []
    for sl, src, tgt in zip(batch["src_lang"], batch["src"], batch["tgt"]):
        tokenizer.src_lang = sl
        tokenizer.tgt_lang = tgt_lang
        enc = tokenizer(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
    return {"input_ids": [e["input_ids"] for e in all_ids],
            "attention_mask": [e["attention_mask"] for e in all_ids],
            "labels": [e["labels"] for e in all_ids]}

def tokenize_single(rows, tgt_lang):
    ds = Dataset.from_list(rows)
    return ds.map(lambda b: nllb_preprocess_single(b, tgt_lang), batched=True, batch_size=16)

tok_combined_train = tokenize_combined(combined_train)
tok_combined_val = tokenize_combined(combined_val)
print(f"combined tokenization complete - {len(tok_combined_train)} train, {len(tok_combined_val)} val")

tok = {lang: {"train": tokenize_single(train_rows, LANGUAGE_CODES[lang]),
             "val": tokenize_single(val_rows, LANGUAGE_CODES[lang])}
       for lang, (train_rows, val_rows, test_rows) in records.items()}
print("per-language tokenization complete -", list(tok.keys()))


Map:   0%|          | 0/58014 [00:00<?, ? examples/s]

Map:   0%|          | 0/4767 [00:00<?, ? examples/s]

combined tokenization complete - 58014 train, 4767 val


Map:   0%|          | 0/19338 [00:00<?, ? examples/s]

Map:   0%|          | 0/1589 [00:00<?, ? examples/s]

Map:   0%|          | 0/19338 [00:00<?, ? examples/s]

Map:   0%|          | 0/1589 [00:00<?, ? examples/s]

Map:   0%|          | 0/19338 [00:00<?, ? examples/s]

Map:   0%|          | 0/1589 [00:00<?, ? examples/s]

per-language tokenization complete - ['kiswahili', 'somali', 'dholuo']


## 6. Zero-shot baseline

The raw base checkpoint, saved once, used as the baseline for every language and both
training approaches.

In [8]:
zeroshot_path = f"{CHECKPOINTS_DIR}/nllb_other_langs_zeroshot"
AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT).save_pretrained(zeroshot_path, safe_serialization=True)
tokenizer.save_pretrained(zeroshot_path)
print(f"zero-shot baseline saved: {zeroshot_path}")


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

zero-shot baseline saved: checkpoints/nllb_other_langs_zeroshot


## 7. Train the combined model first - one run, all three languages at once

In [9]:
_, combined_path, _ = train_stage(
    "nllb_combined_other_langs", "nllb", OTHER_LANG_CFG, "combined_other_langs", CHECKPOINT,
    tok_combined_train, tok_combined_val, tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
print(f"combined checkpoint: {combined_path}")


  nllb_combined_other_langs   arch=nllb  stage=combined_other_langs  init=facebook/nllb-200-distilled-600M  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_combined_other_langs
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,0.501788,0.313890,24.300066,38.218195
2,0.480092,0.284139,26.070866,39.987578
3,0.454249,0.269990,25.102360,37.726875
4,0.437777,0.260605,25.669396,38.163141


nllb_combined_other_langs training time: 68.3 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

combined checkpoint: checkpoints/nllb_combined_other_langs/merged


## 8. Train the independent models - one cell per language

Each starts fresh from the same raw base checkpoint, sees only its own language's data.

In [10]:
kiswahili_path = None
if "kiswahili" in tok:
    _, kiswahili_path, _ = train_stage(
        "nllb_kiswahili_target", "nllb", OTHER_LANG_CFG, "kiswahili_target", CHECKPOINT,
        tok["kiswahili"]["train"], tok["kiswahili"]["val"], tokenizer, MAX_LEN, LORA_KWARGS,
        use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
    print(f"Kiswahili checkpoint: {kiswahili_path}")
else:
    print("Kiswahili training skipped - see Section 2.")


  nllb_kiswahili_target   arch=nllb  stage=kiswahili_target  init=facebook/nllb-200-distilled-600M  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_kiswahili_target
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,0.394739,0.266138,73.836633,84.825968
2,0.320518,0.218049,77.864295,87.284302
3,0.319808,0.197418,79.863531,88.482467
4,0.322926,0.185979,80.368436,88.733042
5,0.281184,0.178372,81.019311,89.149707
6,0.289635,0.174239,81.468284,89.350209
7,0.271344,0.172071,81.552033,89.437626
8,0.293682,0.171081,81.837141,89.677079


nllb_kiswahili_target training time: 21.0 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Kiswahili checkpoint: checkpoints/nllb_kiswahili_target/merged


In [11]:
somali_path = None
if "somali" in tok:
    _, somali_path, _ = train_stage(
        "nllb_somali_target", "nllb", OTHER_LANG_CFG, "somali_target", CHECKPOINT,
        tok["somali"]["train"], tok["somali"]["val"], tokenizer, MAX_LEN, LORA_KWARGS,
        use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
    print(f"Somali checkpoint: {somali_path}")
else:
    print("Somali training skipped - see Section 2.")


  nllb_somali_target   arch=nllb  stage=somali_target  init=facebook/nllb-200-distilled-600M  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_somali_target
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,0.409198,0.235365,69.079328,84.480699
2,0.357654,0.210328,70.555566,85.399524
3,0.364402,0.200553,72.151552,85.998305
4,0.370213,0.194110,72.448694,86.298653
5,0.325387,0.190657,73.115229,86.554057
6,0.329375,0.187654,73.474846,86.720131
7,0.329277,0.186268,73.376099,86.706260
8,0.345931,0.185715,73.426920,86.721040


nllb_somali_target training time: 21.3 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Somali checkpoint: checkpoints/nllb_somali_target/merged


In [12]:
dholuo_path = None
if "dholuo" in tok:
    _, dholuo_path, _ = train_stage(
        "nllb_dholuo_target", "nllb", OTHER_LANG_CFG, "dholuo_target", CHECKPOINT,
        tok["dholuo"]["train"], tok["dholuo"]["val"], tokenizer, MAX_LEN, LORA_KWARGS,
        use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
    print(f"Dholuo checkpoint: {dholuo_path}")
else:
    print("Dholuo training skipped - see Section 2.")


  nllb_dholuo_target   arch=nllb  stage=dholuo_target  init=facebook/nllb-200-distilled-600M  use_lora=True
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_dholuo_target
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,0.703451,0.359677,59.326056,74.911819
2,0.637057,0.325014,61.178108,76.273742
3,0.645723,0.311681,61.732837,76.556786
4,0.634670,0.301822,62.492403,77.037344
5,0.556750,0.296194,63.312362,77.431105
6,0.583604,0.293169,63.147439,77.410990
7,0.567578,0.290876,63.391324,77.750694
8,0.611940,0.290176,63.321962,77.645171


nllb_dholuo_target training time: 23.1 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Dholuo checkpoint: checkpoints/nllb_dholuo_target/merged


In [13]:
paths_independent = {}
if kiswahili_path is not None:
    paths_independent["kiswahili"] = kiswahili_path
if somali_path is not None:
    paths_independent["somali"] = somali_path
if dholuo_path is not None:
    paths_independent["dholuo"] = dholuo_path

print("independent checkpoints:")
for lang, path in paths_independent.items():
    print(f"  {lang:10s} -> {path}")


independent checkpoints:
  kiswahili  -> checkpoints/nllb_kiswahili_target/merged
  somali     -> checkpoints/nllb_somali_target/merged
  dholuo     -> checkpoints/nllb_dholuo_target/merged


## 9. Evaluate everything in one pass - zero-shot, combined, and independent

Every language scored against its own test records, for all three (zero-shot/combined/
independent), overall and by domain, in one combined table - so Section 10 can decide a
winner and Section 11 can report domain ablation without any further generation.

In [14]:
def load_and_generate(path, tgt_code):
    model = AutoModelForSeq2SeqLM.from_pretrained(path)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tok_ = AutoTokenizer.from_pretrained(path)
    def generate(text):
        tok_.src_lang = "eng_Latn"
        enc = tok_(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = tok_.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                             no_repeat_ngram_size=3, repetition_penalty=1.2)
        return tok_.decode(out[0], skip_special_tokens=True)
    return model, generate

eval_results = []
for lang, (_, _, test_rows) in records.items():
    tgt_code = LANGUAGE_CODES[lang]
    runs_to_eval = [("baseline", "zeroshot", zeroshot_path),
                    ("combined", "combined", combined_path),
                    ("independent", lang, paths_independent[lang])]

    for approach, run_name, path in runs_to_eval:
        model, generate = load_and_generate(path, tgt_code)

        preds = [generate(r["src"]) for r in test_rows]
        score = metrics.score(preds, [r["tgt"] for r in test_rows], f"nllb_{approach}_{lang}_overall")
        score.update({"language": lang, "approach": approach, "run": run_name, "domain": "ALL", "n": len(test_rows)})
        eval_results.append(score)

        for domain in DOMAINS:
            subset = [r for r in test_rows if r.get("domain") == domain]
            if not subset:
                continue
            preds_d = [generate(r["src"]) for r in subset]
            score_d = metrics.score(preds_d, [r["tgt"] for r in subset], f"nllb_{approach}_{lang}_{domain}")
            score_d.update({"language": lang, "approach": approach, "run": run_name, "domain": domain, "n": len(subset)})
            eval_results.append(score_d)

        del model
        torch.cuda.empty_cache()

results_df = pd.DataFrame(eval_results)[["language", "approach", "run", "domain", "n", "chrf2pp", "bleu"]]
results_df.to_csv(f"{LOGS_DIR}/nllb_other_languages_all_results.csv", index=False)
print(results_df.to_string(index=False))


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_baseline_kiswahili_overall     chrF2++=72.96  BLEU=55.14
nllb_baseline_kiswahili_Agriculture chrF2++=73.98  BLEU=56.25
nllb_baseline_kiswahili_Health      chrF2++=71.03  BLEU=52.39
nllb_baseline_kiswahili_Education   chrF2++=77.85  BLEU=63.23
nllb_baseline_kiswahili_Security    chrF2++=75.09  BLEU=57.16
nllb_baseline_kiswahili_Governance  chrF2++=71.70  BLEU=53.31


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_combined_kiswahili_overall     chrF2++=74.30  BLEU=54.76
nllb_combined_kiswahili_Agriculture chrF2++=75.14  BLEU=55.27
nllb_combined_kiswahili_Health      chrF2++=73.55  BLEU=53.55
nllb_combined_kiswahili_Education   chrF2++=76.87  BLEU=60.94
nllb_combined_kiswahili_Security    chrF2++=75.52  BLEU=57.37
nllb_combined_kiswahili_Governance  chrF2++=72.01  BLEU=50.69


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

...

nllb_baseline_dholuo_overall        chrF2++=60.20  BLEU=42.10
nllb_baseline_dholuo_Agriculture    chrF2++=64.93  BLEU=50.70
nllb_baseline_dholuo_Health         chrF2++=56.19  BLEU=35.71
nllb_baseline_dholuo_Education      chrF2++=61.19  BLEU=42.23
nllb_baseline_dholuo_Security       chrF2++=59.95  BLEU=39.24
nllb_baseline_dholuo_Governance     chrF2++=62.08  BLEU=39.20


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_combined_dholuo_overall        chrF2++=55.99  BLEU=33.07
nllb_combined_dholuo_Agriculture    chrF2++=52.72  BLEU=31.51
nllb_combined_dholuo_Health         chrF2++=55.16  BLEU=30.92
nllb_combined_dholuo_Education      chrF2++=59.60  BLEU=39.39
nllb_combined_dholuo_Security       chrF2++=59.26  BLEU=34.02
nllb_combined_dholuo_Governance     chrF2++=61.62  BLEU=37.87


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_independent_dholuo_overall     chrF2++=56.39  BLEU=33.67
nllb_independent_dholuo_Agriculture chrF2++=53.52  BLEU=31.92
nllb_independent_dholuo_Health      chrF2++=55.25  BLEU=31.62
nllb_independent_dholuo_Education   chrF2++=60.55  BLEU=40.68
nllb_independent_dholuo_Security    chrF2++=58.86  BLEU=34.12
nllb_independent_dholuo_Governance  chrF2++=62.03  BLEU=38.25
 language    approach       run      domain   n   chrf2pp      bleu
kiswahili    baseline  zeroshot         ALL 985 72.960605 55.143804
kiswahili    baseline  zeroshot Agriculture 262 73.984209 56.250770
kiswahili    baseline  zeroshot      Health 394 71.028477 52.393410
kiswahili    baseline  zeroshot   Education 152 77.846729 63.233119
kiswahili    baseline  zeroshot    Security  66 75.088646 57.155741
kiswahili    baseline  zeroshot  Governance 111 71.702056 53.310947
kiswahili    combined  combined         ALL 985 74.299921 54.756012
kiswahili    combined  combined Agriculture 262 75.137208 55.268705
kiswahili    com

## 10. Decide the winner: combined vs. independent, n-weighted overall

Same principle as the main NLLB ablation's checkpoint selection - weighted by each row's
own `n`, not an unweighted mean, so a language with more test rows counts for what it
actually is.

In [21]:
overall = results_df[(results_df["domain"] == "ALL") & (results_df["approach"] != "baseline")]

weighted = overall.groupby("approach").apply(lambda g: (g["chrf2pp"] * g["n"]).sum() / g["n"].sum())
print("n-weighted overall chrF2++, combined vs. independent (all three languages together):")
print(weighted.round(2).to_string())

print("\nfor reference, per-language breakdown (overall, not yet by domain):")
per_language = overall.pivot_table(index="approach", columns="language", values="chrf2pp")
print(per_language.round(1).to_string())

n-weighted overall chrF2++, combined vs. independent (all three languages together):
approach
combined       67.16
independent    67.23

for reference, per-language breakdown (overall, not yet by domain):
language     dholuo  kiswahili  somali
approach                              
combined       56.0       74.3    71.2
independent    56.4       74.2    71.1


In [22]:
# The n-weighted metric alone favors "independent" by a margin inside normal run-to-run
# noise, and that margin isn't even a consistent per-language effect - see the markdown
# cell below for the full breakdown and justification. WINNING_APPROACH is therefore set
# explicitly here rather than taken as weighted.idxmax()'s literal pick.
WINNING_APPROACH = "combined"
print(f"\nraw n-weighted pick would have been: {weighted.idxmax()!r} ({weighted.round(2).to_dict()})")
print(f"WINNING_APPROACH (explicitly chosen): {WINNING_APPROACH!r} - see justification below")


raw n-weighted pick would have been: 'independent' ({'combined': 67.16, 'independent': 67.23})
WINNING_APPROACH (explicitly chosen): 'combined' - see justification below


## Why we proceed with the combined (multilingual) model

The raw n-weighted metric technically favors **independent** (67.23 vs. 67.16 chrF2++) -
but that 0.07-point gap is well within normal run-to-run noise, and it isn't even a
consistent effect once broken down per language: **combined actually wins two of the
three languages** - Kiswahili (74.3 vs. 74.2) and Somali (71.2 vs. 71.1) - while
independent's entire overall edge comes from Dholuo alone (56.4 vs. 56.0). A 0.1-point win
on two languages against a 0.4-point loss on one isn't a result that should decide the
architecture question by itself.

Combined with that: a single shared-encoder multilingual model is the standard design for
this kind of low-resource multi-target-language setup - Kiswahili, Somali, and Dholuo
reinforce a shared representation of the PSA register through one encoder (every training
row's source side is English regardless of target language, so the same encoder gets
shaped by three decoding objectives at once instead of one), rather than three encoders
that never see each other's training signal at all. Given that the independent approach's
only advantage is a statistically insignificant overall margin driven by a single
language, and the combined approach is both the more standard technique and the simpler
one to deploy and maintain going forward (one checkpoint instead of three), **we proceed
with the combined model** for the domain ablation and demonstration below.

Both sets of checkpoints remain saved either way
(`deployment/nllb/nllb_combined_other_langs/` and
`deployment/nllb/nllb_{kiswahili,somali,dholuo}_target/`) - this choice affects which one
is used for the domain ablation and demo, not what's kept on disk.

## 11. Domain ablation - winning approach only

Filtered from Section 9's results, already computed there - no new generation here.

In [24]:
winner_domain_df = results_df[(results_df["approach"] == WINNING_APPROACH) & (results_df["domain"] != "ALL")]
print(f"chrF2++ by domain, {WINNING_APPROACH} approach, per language:")
domain_summary = winner_domain_df.pivot_table(index="language", columns="domain", values="chrf2pp")
print(domain_summary.round(1).to_string())
domain_summary.to_csv(f"{LOGS_DIR}/nllb_other_languages_winner_domain_summary.csv")

chrF2++ by domain, combined approach, per language:
domain     Agriculture  Education  Governance  Health  Security
language                                                       
dholuo            52.7       59.6        61.6    55.2      59.3
kiswahili         75.1       76.9        72.0    73.6      75.5
somali            69.3       77.0        72.6    70.1      73.3


## 12. Save deployment-ready checkpoints - both approaches, regardless of winner

Nothing gets discarded - the combined model and all three independent models are all
saved, even though only the winner is used for the demo below.

In [25]:
inference.save_for_deployment(combined_path, "nllb", "nllb_combined_other_langs", deployment_dir=DEPLOYMENT_DIR)
for lang, path in paths_independent.items():
    inference.save_for_deployment(path, "nllb", f"nllb_{lang}_target", deployment_dir=DEPLOYMENT_DIR)


deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_combined_other_langs  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_kiswahili_target  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_somali_target  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_dholuo_target  (1 safetensors file(s))


## 13. Demonstration - using the winning approach

In [26]:
samples = [
    "Farmers are urged to prioritize safe agrochemical usage this season.",
    "Ministry of Health: report suspected cholera cases immediately.",
]

demo_paths = ({lang: combined_path for lang in records} if WINNING_APPROACH == "combined"
              else paths_independent)

print(f"Demo using the winning approach: {WINNING_APPROACH}\n")
for lang, path in demo_paths.items():
    demo_model = AutoModelForSeq2SeqLM.from_pretrained(path)
    demo_model.to("cuda" if torch.cuda.is_available() else "cpu")
    demo_tok = AutoTokenizer.from_pretrained(path)
    demo_tok.src_lang = "eng_Latn"
    forced_bos = demo_tok.convert_tokens_to_ids(LANGUAGE_CODES[lang])

    print(f"English to {lang.capitalize()} (PSA-adapted)\n")
    for text in samples:
        enc = demo_tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(demo_model.device)
        out = demo_model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                                  no_repeat_ngram_size=3, repetition_penalty=1.2)
        print(f"[en] {text}")
        print("  ->", demo_tok.decode(out[0], skip_special_tokens=True))
        print()
    del demo_model
    torch.cuda.empty_cache()


Demo using the winning approach: combined



Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

English to Kiswahili (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Wakulima wanahimizwa kuweka kipaumbele matumizi salama ya kemikali za kilimo msimu huu.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Wizara ya Afya: kuripoti visa vya kolari vinavyoshukiwa mara moja.



Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

English to Somali (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Beeraleyda waxaa lagu boorinayaa inay mudnaanta siiyaan isticmaalka kiimikada beeraha ee ammaanka ah xilli ciyaareedkan.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Wasaaradda Caafimaadka: isla markiiba soo sheeg kiisaska laga shakiyo ee cudurka kolera.



Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

English to Dholuo (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Ijiwo jopur mondo oket mokwongo tiyo gi gik ma nigi tuoche mag yien e kindewagi.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Ministry of Health: nyis mapiyo ni nitie joma nigi tuwo mar cholera.

